# TD-clip retrain — sweep runs over seeds × del-lims (wandb)

Retrains pretrained checkpoints under a grid of **seeds × TD clips** (`limit_delta`,
here called *del-lim*). Each (seed, del-lim) is a separate wandb run named
`Retrain_Seed{seed}_{tag}` (mirroring training's `Run_Seed{seed}`), retrained with
the **same hyperparameters as `run_training.ipynb`**, and saved as a full checkpoint
bundle under `results/retrain/<del_lim>/checkpoints_retrain_<seed>/`.

This notebook only **runs & saves** — plotting/averaging/inference live in the
companion plotting notebook.

In [5]:
# ── Imports ───────────────────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import torch

from agents.ppo import PPOAgent

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    warnings.warn('wandb not installed — logging disabled.')

print('Imports OK  |  wandb available:', WANDB_AVAILABLE)

Imports OK  |  wandb available: True


## Config

In [ ]:
# ── Sweep grid: seeds × del-lims (EDIT THESE) ─────────────────────────────
SEEDS    = [0, 7, 64, 27, 42, 100, 107, 1997, 2003, 2026]                       # seeds that have results/checkpoints_<seed>/
# DEL_LIMS = [None, -5e-3, -3e-3, -1e-3, -5e-4, -3e-4,-1e-4, -5e-5, -3e-5, -1e-5, -5e-6, -3e-6,
#             -1e-6, 1e-6, 3e-6, 5e-6, 1e-5, 3e-5, 5e-5, 1e-4, 3e-4, 5e-4, 1e-3, 3e-3, 5e-3 ] #[None, -1e-5, 1e-5]       # limit_delta values. None = no clip.
                                     #   +c upper clip (caps +TD), -c lower clip (floors -TD)
DEL_LIMS = [2e-1]#[5e-2, 1e-1]
CKPT_NAME        = 'last'            # source checkpoint per seed: 'last' | 'best' | 'epN'
RETRAIN_EPISODES = 500

# ── Retrain hyperparameters — MATCHED to run_training.ipynb ────────────────
TRAIN_KW = dict(
    rollout_steps      = 256,
    ppo_epochs         = 4,
    minibatch_size     = 64,
    shared_ac_lr       = 1e-4,
    # lr_schedule        = {'type': 'exponential', 'final_frac': 0.1},
    log_every_episodes = 10,
)

# ── wandb ─────────────────────────────────────────────────────────────────
LOG_WANDB    = True
PROJECT_NAME = 'Food RL'             # same project as training
RETRAIN_ROOT = 'results/retrain_500ep'
os.makedirs(RETRAIN_ROOT, exist_ok=True)

if LOG_WANDB and not WANDB_AVAILABLE:
    LOG_WANDB = False
    print('wandb not available — LOG_WANDB forced False.')

# ── del-lim -> readable tag / folder name ─────────────────────────────────
def dl_tag(dl):
    if dl is None or dl == 0:
        return 'noclip'
    return f'{"pos" if dl > 0 else "neg"}_{abs(float(dl)):g}'

print(f'Seeds    : {SEEDS}')
print(f'Del-lims : {DEL_LIMS}  ->  tags {[dl_tag(d) for d in DEL_LIMS]}')
print(f'Grid     : {len(SEEDS) * len(DEL_LIMS)} runs x {RETRAIN_EPISODES} eps')
print(f'Save root: {RETRAIN_ROOT}/<del_lim>/checkpoints_retrain_<seed>/')

Seeds    : [0, 7, 64, 27, 42, 100, 107, 1997, 2003, 2026]
Del-lims : [0.1]  ->  tags ['pos_0.1']
Grid     : 10 runs x 500 eps
Save root: results/retrain_500ep/<del_lim>/checkpoints_retrain_<seed>/


## Run the sweep

In [7]:
# ── Retrain every (del_lim, seed) ─────────────────────────────────────────
import time

def retrain_one(seed, dl):
    src_meta = f'results/training_checkpoints/checkpoints_{seed}/ppo_agent_{CKPT_NAME}_meta.json'
    if not os.path.exists(src_meta):
        print(f'  SKIP seed {seed}: no checkpoint at {src_meta}')
        return None

    agent = PPOAgent.from_checkpoint(src_meta, device='cpu')
    agent.args['limit_delta'] = dl                     # apply the del-lim (clip)

    run = None
    if LOG_WANDB:
        run = wandb.init(
            project=PROJECT_NAME,
            name=f'Retrain_500ep_Seed{seed}_{dl_tag(dl)}',
            reinit=True,
            config={'seed': seed, 'limit_delta': dl, 'del_lim_tag': dl_tag(dl),
                    'source_ckpt': src_meta, 'retrain_episodes': RETRAIN_EPISODES,
                    **TRAIN_KW},
        )

    ckpt_dir = os.path.join(RETRAIN_ROOT, dl_tag(dl), 'checkpoints_retrain')
    
    tlog = agent.train(num_episodes=RETRAIN_EPISODES, log_wandb=LOG_WANDB,
                           printing=False, checkpoint_every=100, checkpoint_dir=ckpt_dir,
                           **TRAIN_KW)

    # ── save bundle: results/retrain/<del_lim>/checkpoints_retrain_<seed>/ ──
    # save_final appends '_<seed>' to the dir, giving the exact folder we want.
    ckpt_dir = os.path.join(RETRAIN_ROOT, dl_tag(dl), 'checkpoints_retrain')
    prefix   = agent.save_final(ckpt_dir, log_wandb=LOG_WANDB)      # .../checkpoints_retrain_<seed>/ppo_agent_last_*
    out_dir  = os.path.dirname(prefix)
    agent.save_training_log(os.path.join(out_dir, 'training_log.npz'))
    with open(os.path.join(out_dir, 'retrain_info.json'), 'w') as f:
        json.dump({'seed': seed, 'limit_delta': dl, 'del_lim_tag': dl_tag(dl),
                   'source_ckpt': src_meta, 'retrain_episodes': RETRAIN_EPISODES,
                   'train_kw': TRAIN_KW}, f, indent=2)
    if run is not None:
        wandb.finish()
    return {'seed': seed, 'dl': dl, 'out_dir': out_dir,
            'last_return': float(tlog['reward'][-1]),
            'roll50_return': float(np.mean(tlog['reward'][-50:])),
            'roll50_food': float(np.mean(tlog['consumption'][-50:]))}

summary = []
t_all = time.time()
for dl in DEL_LIMS:
    for seed in SEEDS:
        print(f'=== del_lim={dl} ({dl_tag(dl)})  |  seed {seed} ===')
        t0 = time.time()
        res = retrain_one(seed, dl)
        if res is None:
            continue
        summary.append(res)
        print(f'  [{time.time()-t0:5.1f}s] roll50 return={res["roll50_return"]:8.2f}  '
              f'food={res["roll50_food"]:5.2f}  -> {res["out_dir"]}')
print(f'\nSweep done in {time.time()-t_all:.1f}s — {len(summary)} runs saved under {RETRAIN_ROOT}/')

=== del_lim=0.1 (pos_0.1)  |  seed 0 ===
[FoodEnv] Loading  'glucose'  from  'food_dataset/serum_glucose.csv'
           min=70.0758  max=142.5123   foods=19   time_points=500
[FoodEnv] Loading  'peptides'  from  'food_dataset/small_peptides_absorbed.csv'
           min=0.0000  max=0.0018   foods=19   time_points=500
[FoodEnv] Loading  'fatty_acids'  from  'food_dataset/fatty_acids_absorbed.csv'
           min=0.0000  max=0.0007   foods=19   time_points=500
[FoodEnv] Ready — 19 foods | 3 time-series nutrients | 0 cumulative nutrients
[FoodEnv] Loading shadow nutrient 'fullness'  from  'food_dataset/fullness.csv'
[FoodEnv] Loading shadow nutrient 'hunger'  from  'food_dataset/hunger.csv'
[FoodEnv] Loading shadow nutrient 'cck'  from  'food_dataset/cck.csv'
[FoodEnv] Loading shadow nutrient 'ghrelin'  from  'food_dataset/ghrelin.csv'
[FoodEnv] Loading shadow nutrient 'glp_1'  from  'food_dataset/glp_1.csv'
[FoodEnv] Loading shadow nutrient 'pyy'  from  'food_dataset/pyy.csv'
[FoodEnv] Sh

train/actor_loss,▅▅▅▅▄▃▅▄▄▄▄█▅▄▄▅▆▃▄▅▇▆▄▄▆▃▄▅▄▃▅▄▆▅▅▄▁▃▅▅
train/consumption,▁▃▃▆▆▅▆█▆▆█▃▅▄▅▄▅▆█▂▂▂▂▇▃▃▂▂▄▃▃▅▃▆▄▅▆▃▆▅
train/critic_loss,▂▂▂▁▂▂█▁▂▁▁▁▂▁▂▂▁▂▁▂▁▂▄▂▁▂▂▁▂▂▃▁▄▂▁▂▂▁▁▂
train/distance,▂▁▂▂▆▄█▅▅▅▂▅▅▃▄▅▄▅▆▃▄▂▄▆▃▃▂▂▃▃▂▄▁▂▃▂▄▁▃▅
train/entropy,▃▁▃▅▄▃▄▆▇▃▇▃▅▁▆▆▄▇▇▇▅▂▅▆▇▅▅▁▄▂▄▂▆▃▃█▅▂▆▅
train/reward,▇█▇▇▃▄▁▁▅▄▂▆▃▄▅▅▅▅▄▄▄▇▅▆▅▅▃▇▆▇▅▆▇▄█▆▆▆▆▅
train/reward_rolling_avg,███▇▇▄▃▂▁▁▁▃▃▃▃▃▄▄▅▅▅▅▆▆▅▅▅▇▇▇▇▇▇▇▇▇▆▆▇▆
train/shared_ac_lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/actor_loss,-0.01259
train/consumption,18
train/critic_loss,0.60214


  [ 85.5s] roll50 return=  149.22  food=17.78  -> results/retrain_500ep/pos_0.1/checkpoints_retrain_0
=== del_lim=0.1 (pos_0.1)  |  seed 7 ===
[FoodEnv] Loading  'glucose'  from  'food_dataset/serum_glucose.csv'
           min=70.0758  max=142.5123   foods=19   time_points=500
[FoodEnv] Loading  'peptides'  from  'food_dataset/small_peptides_absorbed.csv'
           min=0.0000  max=0.0018   foods=19   time_points=500
[FoodEnv] Loading  'fatty_acids'  from  'food_dataset/fatty_acids_absorbed.csv'
           min=0.0000  max=0.0007   foods=19   time_points=500
[FoodEnv] Ready — 19 foods | 3 time-series nutrients | 0 cumulative nutrients
[FoodEnv] Loading shadow nutrient 'fullness'  from  'food_dataset/fullness.csv'
[FoodEnv] Loading shadow nutrient 'hunger'  from  'food_dataset/hunger.csv'
[FoodEnv] Loading shadow nutrient 'cck'  from  'food_dataset/cck.csv'
[FoodEnv] Loading shadow nutrient 'ghrelin'  from  'food_dataset/ghrelin.csv'
[FoodEnv] Loading shadow nutrient 'glp_1'  from  'food

train/actor_loss,▆█▆▇▆▇▅▅▆▇▆▇▇█▆█▇▆▅▇▆▆▆▆█▆▄▆▁▇▇▆█▆▇█▅▄▅▇
train/consumption,▄▁▂▁▂▂▅▄▅▅▇▃▅▃▃▆▅▂▅▃▅▅█▄▄▄▃▂▆▃▅▆▅▃▅▃▂▂▄▃
train/critic_loss,▃▂▃▄▁▁▃▄▃▁█▃▃▂▆▁▄▃▁▃▃▅▁▂▃▃▃▃▄▂▁▄▃▄▂▂▂▂▃▇
train/distance,▇▃▃▄▂▃▆▂▅▅▄▅▅▄▄▃▃▄▁▂▄█▅▅▅▄▃▃▅▂▂▂▃▃▃▅▄▂▄▅
train/entropy,▃▂▇▅▅▆▇▆▂▁▄▃▃▃▇▂▅▄▄▅▄▅▄▄▅▅▆▆█▆▄▃█▃▅▄▅▅▂▂
train/reward,▂▅▅▅▆▇▄▇▄▄▁▄▄▃▄▃▆▄█▆▅▁▄▂▄▄▆▇▃▇▆▆▆▅▇▃▅▇▅▃
train/reward_rolling_avg,▂▇▆▆█▆▇▇▇▆▅▃▁▁▁▂▁▄▄▅▄▂▂▂▄▅▄▅▆▆█▇▆▇█▅▃▃▂▂
train/shared_ac_lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/actor_loss,-0.00155
train/consumption,12
train/critic_loss,1.2038


  [ 84.4s] roll50 return=  161.29  food=12.18  -> results/retrain_500ep/pos_0.1/checkpoints_retrain_7
=== del_lim=0.1 (pos_0.1)  |  seed 64 ===
[FoodEnv] Loading  'glucose'  from  'food_dataset/serum_glucose.csv'
           min=70.0758  max=142.5123   foods=19   time_points=500
[FoodEnv] Loading  'peptides'  from  'food_dataset/small_peptides_absorbed.csv'
           min=0.0000  max=0.0018   foods=19   time_points=500
[FoodEnv] Loading  'fatty_acids'  from  'food_dataset/fatty_acids_absorbed.csv'
           min=0.0000  max=0.0007   foods=19   time_points=500
[FoodEnv] Ready — 19 foods | 3 time-series nutrients | 0 cumulative nutrients
[FoodEnv] Loading shadow nutrient 'fullness'  from  'food_dataset/fullness.csv'
[FoodEnv] Loading shadow nutrient 'hunger'  from  'food_dataset/hunger.csv'
[FoodEnv] Loading shadow nutrient 'cck'  from  'food_dataset/cck.csv'
[FoodEnv] Loading shadow nutrient 'ghrelin'  from  'food_dataset/ghrelin.csv'
[FoodEnv] Loading shadow nutrient 'glp_1'  from  'foo

...

  [ 84.7s] roll50 return=  158.85  food=13.90  -> results/retrain_500ep/pos_0.1/checkpoints_retrain_1997
=== del_lim=0.1 (pos_0.1)  |  seed 2003 ===
[FoodEnv] Loading  'glucose'  from  'food_dataset/serum_glucose.csv'
           min=70.0758  max=142.5123   foods=19   time_points=500
[FoodEnv] Loading  'peptides'  from  'food_dataset/small_peptides_absorbed.csv'
           min=0.0000  max=0.0018   foods=19   time_points=500
[FoodEnv] Loading  'fatty_acids'  from  'food_dataset/fatty_acids_absorbed.csv'
           min=0.0000  max=0.0007   foods=19   time_points=500
[FoodEnv] Ready — 19 foods | 3 time-series nutrients | 0 cumulative nutrients
[FoodEnv] Loading shadow nutrient 'fullness'  from  'food_dataset/fullness.csv'
[FoodEnv] Loading shadow nutrient 'hunger'  from  'food_dataset/hunger.csv'
[FoodEnv] Loading shadow nutrient 'cck'  from  'food_dataset/cck.csv'
[FoodEnv] Loading shadow nutrient 'ghrelin'  from  'food_dataset/ghrelin.csv'
[FoodEnv] Loading shadow nutrient 'glp_1'  from 

train/actor_loss,▆▇▆▇▆▇▇▇█▆▇▇▇▇▁▅▇▆▆▇█▆▆▆▆▇▇▇▇▇▆▆▆▇▇▇▇▆▃▅
train/consumption,▃▂▅▄▄▃▃▅▃▂▄▁▃▃▅▅▅▄▄▃▄▅▅▄▃▅▆▄▅▆▃▅▄▅▆█▅▄▆▅
train/critic_loss,▁▄▂▂▂█▄▅▄▄▂▄▂▃▁▂▂▅▇▃▂▄▂▄▂▆▂▅▂▆▄▆▃▁▃▄▆▅▃▁
train/distance,▁▅▅▃▆▄▃▁▄▂▅▁▂▄▁▂▄▂▄▅▄▃▄▃▃█▅▆▄▅▄▂▃▂▅▅▃▅▆▃
train/entropy,▄▅▂▇▃▃▄▇▃▁▆▄▄█▇▆█▅▅▅▄▅▆▅▅▆▁▄▃▃▅▄▅▇▄▅▆▃▇▄
train/reward,▇▄▃▆▁▅▂▆█▆▂▇▅▄▇▆▅▇▅▃▄▆▄▅▅▁▃▂▅▂▅▅▅▇▄▃▃▅▄▇
train/reward_rolling_avg,▇▇█▇▆▄▃▄▅▆▇▆▆▅▅▆▅▄▅▅▆▆▅▄▃▄▄▄▅▅▂▂▂▄▃▂▁▂▂▃
train/shared_ac_lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/actor_loss,-0.00507
train/consumption,14
train/critic_loss,0.18517


  [ 85.3s] roll50 return=  159.65  food=13.64  -> results/retrain_500ep/pos_0.1/checkpoints_retrain_2003
=== del_lim=0.1 (pos_0.1)  |  seed 2026 ===
[FoodEnv] Loading  'glucose'  from  'food_dataset/serum_glucose.csv'
           min=70.0758  max=142.5123   foods=19   time_points=500
[FoodEnv] Loading  'peptides'  from  'food_dataset/small_peptides_absorbed.csv'
           min=0.0000  max=0.0018   foods=19   time_points=500
[FoodEnv] Loading  'fatty_acids'  from  'food_dataset/fatty_acids_absorbed.csv'
           min=0.0000  max=0.0007   foods=19   time_points=500
[FoodEnv] Ready — 19 foods | 3 time-series nutrients | 0 cumulative nutrients
[FoodEnv] Loading shadow nutrient 'fullness'  from  'food_dataset/fullness.csv'
[FoodEnv] Loading shadow nutrient 'hunger'  from  'food_dataset/hunger.csv'
[FoodEnv] Loading shadow nutrient 'cck'  from  'food_dataset/cck.csv'
[FoodEnv] Loading shadow nutrient 'ghrelin'  from  'food_dataset/ghrelin.csv'
[FoodEnv] Loading shadow nutrient 'glp_1'  from 

train/actor_loss,▅▆▆▁▆▅▅▇▇▇▃▅▅▆▅▆▆▆▃▄▆▄▅▅▅▇▇▇█▆▆▄▆▃▇▇▅▆▆▄
train/consumption,▄▆▁▃▅▂▄▄▅▅▄▄▃▃▅▄▅▅▄▄▅▅▃▅▄▄▄▄▂▅▄▄█▆▅▄▄▅▄▃
train/critic_loss,▄▄▂▅▃▂▃▅▄▃▂▄▅▂▂▂▄▆▃▂▅▂▄▃▅▆█▃▂▂▃▂▃▃▃▅▂▂▂▁
train/distance,▃▅▄█▄▄▄▆▅▂▃▃▂▃▄▅▇▇▂▃▇▅▃▄▇▁▆▅▄▅▆▆▇▅▂▄▄▃▅▄
train/entropy,█▅▄█▆▂▄▃▁▂▄▃▄▅▄▃▂▅▂▄▆▃▇▆▄▃▁▁▂▂▂▃▂▂▃▃▄▂▄▅
train/reward,▅▃▅▁▆▅▅▂▃▆▇█▆▆▆▂▂▅▆▃▅▃▄▁▄▃▅▆▃▄▃▂▂▄▇▅▆▅▃▅
train/reward_rolling_avg,██▇▇▇▄▄▃▂▃▄▄▅▅▄▅▄▄▅▄▄▄▃▃▃▄▄▅▄▃▂▁▁▁▂▄▅▅▆▆
train/shared_ac_lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/actor_loss,-0.00494
train/consumption,11
train/critic_loss,0.07799


  [ 86.1s] roll50 return=  168.15  food=12.62  -> results/retrain_500ep/pos_0.1/checkpoints_retrain_2026

Sweep done in 847.9s — 10 runs saved under results/retrain_500ep/


In [8]:
# ── Recap table of what was saved ─────────────────────────────────────────
import pandas as pd
if summary:
    df = pd.DataFrame(summary)
    df['del_lim_tag'] = [dl_tag(d) for d in df['dl']]
    display(df[['del_lim_tag', 'seed', 'roll50_return', 'roll50_food', 'out_dir']].round(3))
else:
    print('No runs saved — check SEEDS have results/checkpoints_<seed>/ bundles.')

,del_lim_tag,seed,roll50_return,roll50_food,out_dir
0,pos_0.1,0,149.216,17.78,results/retrain_500ep/pos_0.1/checkpoints_retr...
1,pos_0.1,7,161.289,12.18,results/retrain_500ep/pos_0.1/checkpoints_retr...
2,pos_0.1,64,156.306,15.86,results/retrain_500ep/pos_0.1/checkpoints_retr...
3,pos_0.1,27,162.201,12.86,results/retrain_500ep/pos_0.1/checkpoints_retr...
4,pos_0.1,42,152.186,19.50,results/retrain_500ep/pos_0.1/checkpoints_retr...
5,pos_0.1,100,152.398,14.38,results/retrain_500ep/pos_0.1/checkpoints_retr...
6,pos_0.1,107,157.071,15.56,results/retrain_500ep/pos_0.1/checkpoints_retr...
7,pos_0.1,1997,158.854,13.90,results/retrain_500ep/pos_0.1/checkpoints_retr...
8,pos_0.1,2003,159.646,13.64,results/retrain_500ep/pos_0.1/checkpoints_retr...
9,pos_0.1,2026,168.148,12.62,results/retrain_500ep/pos_0.1/checkpoints_retr...
